# Restructuration TF-IDF — fil conducteur

Vous reprenez votre classifieur d'intentions du **Module 1** et le
restructurez dans l'architecture du **Module 4**. **Complétez d'abord les TODO de `src/`**, puis ce notebook orchestre les
modules de `src/` ; il ne contient pas la logique, il l'appelle.


In [1]:
%load_ext autoreload
%autoreload 2
    
# Amorçage : se placer à la racine du projet (dossier contenant src/)
import os, sys
while not os.path.isdir('src'):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        break
    os.chdir(parent)
sys.path.insert(0, os.getcwd())
print('cwd =', os.getcwd())

cwd = /home/a453784/simplon-briefs/Brief_Estival_PipelineTF-IDF


## Étape 1 — Charger les données

On regarde ce que produit le chargement : quelles colonnes a-t-on sous la main ?

In [2]:
from src import data_loading, config
df = data_loading.load()
print(df.columns.tolist())
df.head()

[data_loading] Banking77 chargé : 13083 lignes
['text', 'category']


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


## Étape 2 — La configuration centralisée

Tous les réglages vivent dans `config.py` : quelle colonne est le texte (l'entrée du modèle), laquelle est la cible, la graine, le découpage. Plus aucune valeur codée en dur ailleurs — on sait où regarder.

In [3]:
print('Texte  :', config.TEXT_COL)
print('Cible  :', config.LABEL_COL)
print('Graine :', config.RANDOM_SEED)
print('Test   :', config.TEST_SIZE)

Texte  : text
Cible  : category
Graine : 42
Test   : 0.25


## Étape 3 — Découpage stratifié + TF-IDF

Découpage stratifié sur l'intention (préserve les classes rares). Le TF-IDF sera ajusté sur le train uniquement — c'est le Pipeline qui le garantit.

In [4]:
from src import features
X_train, X_test, y_train, y_test = features.make_split(df)
print('train:', len(X_train), '| test:', len(X_test))
print('classes:', y_train.nunique())

train: 9812 | test: 3271
classes: 77


## Étape 4 — Les candidats

Au Module 1 : un seul modèle. Ici, au moins trois familles, comparées.

In [5]:
from src import models
candidates = models.build_candidates()
list(candidates.keys())

['logreg', 'linear_svm', 'naive_bayes']

## Étape 5 — Les métriques

Multi-classe déséquilibré : on regarde le macro-F1 et la précision balancée, pas seulement l'exactitude.

In [6]:
from src import evaluation
pipe = candidates['logreg'].fit(X_train, y_train)
y_pred = pipe.predict(X_test)
evaluation.evaluate(y_test, y_pred)

{'accuracy': 0.8615102415163558,
 'balanced_accuracy': 0.8588135328010937,
 'f1_macro': 0.8601988281123308,
 'f1_weighted': 0.861219984913214,
 'recall_macro': 0.8588135328010937,
 'recall_per_class': {'Refund_not_showing_up': 0.941,
  'activate_my_card': 0.88,
  'age_limit': 1.0,
  'apple_pay_or_google_pay': 0.929,
  'atm_support': 0.875,
  'automatic_top_up': 0.857,
  'balance_not_updated_after_bank_transfer': 0.83,
  'balance_not_updated_after_cheque_or_cash_deposit': 0.964,
  'beneficiary_not_allowed': 0.857,
  'cancel_transfer': 1.0,
  'card_about_to_expire': 0.976,
  'card_acceptance': 0.88,
  'card_arrival': 0.833,
  'card_delivery_estimate': 0.684,
  'card_linking': 0.978,
  'card_not_working': 0.842,
  'card_payment_fee_charged': 0.842,
  'card_payment_not_recognised': 0.885,
  'card_payment_wrong_exchange_rate': 0.865,
  'card_swallowed': 1.0,
  'cash_withdrawal_charge': 0.926,
  'cash_withdrawal_not_recognised': 0.86,
  'change_pin': 0.854,
  'compromised_card': 0.839,
  'co

## Étape 6 — Benchmark complet

Le geste du Module 4 : tout le monde sur la même grille, résultats dans `outputs/results.csv`.

In [7]:
from src.benchmark import run
results = run()
results

[data_loading] Banking77 chargé : 13083 lignes
[benchmark] logreg           macro_f1=0.860  acc=0.862
[benchmark] linear_svm       macro_f1=0.894  acc=0.894
[benchmark] naive_bayes      macro_f1=0.827  acc=0.836
[benchmark] résultats écrits dans outputs/results.csv


,accuracy,balanced_accuracy,f1_macro,f1_weighted,recall_macro,recall_per_class
model,,,,,,
linear_svm,0.893611,0.893308,0.893592,0.893463,0.893308,"{'Refund_not_showing_up': 0.961, 'activate_my_..."
logreg,0.861510,0.858814,0.860199,0.861220,0.858814,"{'Refund_not_showing_up': 0.941, 'activate_my_..."
naive_bayes,0.835830,0.821432,0.827204,0.833600,0.821432,"{'Refund_not_showing_up': 0.922, 'activate_my_..."


## Étape 7 — Lire le déséquilibre

Le rappel par classe révèle ce que l'exactitude globale masque : les intentions rares sont-elles ratées ?

In [8]:
rep = evaluation.per_class_report(y_test, y_pred)
rep[['recall','support']].sort_values('support').round(2)

,recall,support
accuracy,0.86,0.86
contactless_not_working,0.74,19.00
virtual_card_not_working,0.65,20.00
card_acceptance,0.88,25.00
card_swallowed,1.00,25.00
...,...,...
wrong_amount_of_cash_received,0.93,55.00
direct_debit_payment_not_recognised,0.82,56.00
card_payment_fee_charged,0.84,57.00
macro avg,0.86,3271.00


## Étape 8 — Conclure

En une cellule markdown : quel modèle retenez-vous, sur quelle métrique, et quel défaut de dette technique avez-vous corrigé au passage (l'ajustement du TF-IDF sur le train uniquement) ?

Le modèle retenu est Linear SVM, qui obtient les meilleures performances sur l'ensemble des métriques évaluées. Son F1-score macro atteint 0,894, contre 0,860 pour Logistic Regression et 0,827 pour Naive Bayes.
Le choix est néanmoins fondé principalement sur le F1 Macro, métrique particulièrement adaptée à la classification multi-classes puisqu'elle accorde le même poids à chaque intention et permet d'évaluer la capacité du modèle à bien traiter l'ensemble des classes.
Les résultats montrent également que le SVM obtient le meilleur recall macro (0,893) et les meilleures performances sur de nombreuses intentions individuelles. Il constitue donc le meilleur compromis entre précision et rappel pour ce problème de classification d'intentions textuelle